# Wordmaze × GRPO — Qwen3.5-4B instruct, thinking on, LoRA

Runner for `wordmaze/GUIDE.md`. **Do the baseline table (Step 1) before training** — GRPO can only amplify a non-zero solve rate.

**Runtime → A100.** Steps are slow (16 × ~1.5k-token completions); use `--resume` across sessions.

In [ ]:
!nvidia-smi
import torch; print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

In [ ]:
%%capture
!pip install -U "trl>=1.11" "transformers>=5.2" datasets peft accelerate wandb matplotlib
!pip install flash-linear-attention          # Qwen3.5 Gated-DeltaNet kernels (Triton). Without it: silent slow path.
!pip install wordfreq
# Optional & minor (often fails to build on Colab; skip if so): !pip install causal-conv1d --no-build-isolation
!python -c "from fla.ops.gated_delta_rule import chunk_gated_delta_rule; print('fla OK')"   # must print fla OK
# Optional fast generation:     !pip install "vllm==0.27.1"   -> then add --use_vllm to the train command

In [ ]:
# Get the code. EITHER clone your repo ...
REPO_URL = "https://github.com/<you>/grpo_puzzles"   # <- fill in after you push this folder
# !git clone $REPO_URL
# ... OR upload the .py files from this folder with the Colab file browser into /content/grpo_puzzles/

In [ ]:
from huggingface_hub import login; login()   # push results later
import wandb; wandb.login()                        # the curves are the deliverable

In [ ]:
%cd /content/grpo_puzzles/wordmaze
!python verifier.py | tail -3
!python rewards.py | tail -1
!python data.py | tail -1

## Step 1 — baselines on the easiest bucket (4-letter, 3 moves). ~10–30 min per model.

In [ ]:
for model, tag in [("Qwen/Qwen3.5-4B", "4b"), ("Qwen/Qwen3.5-2B", "2b"), ("Qwen/Qwen3.5-0.8B", "08b")]:
    !python eval_wordmaze.py --model $model --tag {tag}-base-4x3 --word_lengths 4 --move_counts 3
!python eval_wordmaze.py --model Qwen/Qwen3.5-4B --tag 4b-base-4x3-spaced --word_lengths 4 --move_counts 3 --spaced

In [ ]:
import json, glob
print(f"{'tag':28s} fully_valid  valid_ladder  one_letter  password  answered  trunc")
for p in sorted(glob.glob("results/*-base-*.json")):
    r = json.load(open(p))["metrics"]["rates"]
    print(f"{p[8:-5]:28s} {r['fully_valid']:.3f}        {r['valid_ladder']:.3f}         {r['one_letter_changes']:.3f}       {r['password_matches']:.3f}     {r['answered']:.3f}     {r['truncated_no_answer']:.3f}")

## Step 2 — train (default: 4B, LoRA, 4×3 bucket, 200 steps). Decide the model from the table above (GUIDE §5).

In [ ]:
RUN_DIR = "runs/wordmaze-qwen3.5-4b-lora"
!python train_wordmaze_grpo.py --output_dir $RUN_DIR --max_steps 200 --report_to wandb --run_name wm-4b-4x3
# resume later:  !python train_wordmaze_grpo.py --output_dir $RUN_DIR --max_steps 400 --resume $RUN_DIR/checkpoint-200
# ablations:     --spaced      --model Qwen/Qwen3.5-2B      --no_thinking --max_completion_length 256

## Step 3 — curves

In [ ]:
KEYS = ["rewards/solve_reward/mean", "rewards/ladder_reward/mean", "completions/clipped_ratio"]
TITLES = ["solve reward (fully_valid)", "ladder reward (valid ladder, any password)", "truncated fraction"]
import json, glob, os
import matplotlib.pyplot as plt

def latest_state(run_dir):
    ck = sorted(glob.glob(f"{run_dir}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    path = os.path.join(ck[-1], "trainer_state.json") if ck else os.path.join(run_dir, "final", "trainer_state.json")
    return json.load(open(path))["log_history"]

hist = latest_state(RUN_DIR)
def series(key):
    return [(h["step"], h[key]) for h in hist if key in h]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, key, title in zip(axes, KEYS, TITLES):
    s = series(key)
    if s:
        ax.plot(*zip(*s)); ax.set_title(title); ax.set_xlabel("optimizer step"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{RUN_DIR}/curves.png", dpi=150); plt.show()
print("saved", f"{RUN_DIR}/curves.png")

## After — same bucket/split as before, plus unseen buckets (generalisation) and the published test split

In [ ]:
!python eval_wordmaze.py --model Qwen/Qwen3.5-4B --adapter $RUN_DIR/final --tag 4b-grpo-4x3 --word_lengths 4 --move_counts 3
!python eval_wordmaze.py --model Qwen/Qwen3.5-4B --adapter $RUN_DIR/final --tag 4b-grpo-5x3 --word_lengths 5 --move_counts 3
!python eval_wordmaze.py --model Qwen/Qwen3.5-4B --adapter $RUN_DIR/final --tag 4b-grpo-4x4 --word_lengths 4 --move_counts 4
!python eval_wordmaze.py --model Qwen/Qwen3.5-4B --adapter $RUN_DIR/final --tag 4b-grpo-m34-test --split test   # vs published 18.5%

In [ ]:
import json
for tag in ["4b-base-4x3", "4b-grpo-4x3"]:
    m = json.load(open(f"results/{tag}.json"))["metrics"]
    print(tag, {k: round(v, 3) for k, v in m["rates"].items() if k in ("fully_valid", "valid_ladder", "one_letter_changes", "password_matches", "all_valid_words")})

## Publish
Adapter → Hub (`PeftModel.push_to_hub`), results JSONs + `curves.png` → repo, thread → tag @immortaldip. GUIDE §9 lists the five artefacts that make this the missing "part two".